## Install package

In [28]:
!pip install -q joblib
!pip install -q streamlit

## Load package

In [29]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib

## Load data

In [41]:
#load disease_data csv file

from google.colab import drive
drive.mount('/content/gdrive')
data = pd.read_csv('/content/gdrive/MyDrive/Omdena/NLP in drug prediction/test/processed_diseases-priority.csv')

# Select only 100 unique diseases
#selected_diseases = data['Disease'].unique()[:100]  # Pick the first 100 unique diseases
#data = data[data['Disease'].isin(selected_diseases)]
#data.to_csv('/content/symptoms_data.csv', index = False)

# Set target sample size per disease
target_sample_size = 3

# Perform random oversampling to ensure each disease
df = data.groupby('Disease', group_keys=False).apply(lambda x: x.sample(target_sample_size, replace=True))

# Reset index
df= df.reset_index(drop=True)

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


<ipython-input-41-e64d27284021>:16: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = data.groupby('Disease', group_keys=False).apply(lambda x: x.sample(target_sample_size, replace=True))


## Model building

In [54]:
# Convert symptoms to a structured format using tfidf
vectorizer_tfidf = TfidfVectorizer(max_features=2000)
X = vectorizer_tfidf.fit_transform(df["Symptoms"])  # Convert symptoms into a numerical format

# Encode the target variable (disease)
y = df["Disease"].str.lower()

# Split the dataset into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train a Random Forest classifier
clf = RandomForestClassifier(n_estimators=15, random_state=42)
clf.fit(X_train, y_train)

# Predict on the test set
y_pred = clf.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(accuracy)


0.7998610145934677


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_

## Export RF model and vectorizer_tfidf

In [55]:
###Export RF model and vectorizer_tfidf
joblib.dump(clf, "/content/disease_model.pkl")
joblib.dump(vectorizer_tfidf, "/content/tfidf_vectorizer.pkl")

['/content/tfidf_vectorizer.pkl']

## Creating the app

In [58]:
%%writefile app.py
import streamlit as st
import pandas as pd
import joblib
import os
import requests

# 🔹 Set Google Drive path (Modify if necessary)
drive_path = "/content/"


# 🔹 Load the trained model and vectorizer
@st.cache_resource
def load_model():
    model_path = os.path.join(drive_path, "disease_model.pkl")
    vectorizer_path = os.path.join(drive_path, "tfidf_vectorizer.pkl")

    if not os.path.exists(model_path) or not os.path.exists(vectorizer_path):
        st.error("🚨 Model or vectorizer not found! Please train and save them first.")
        return None, None

    clf = joblib.load(model_path,mmap_mode="r")
    vectorizer = joblib.load(vectorizer_path)
    return clf, vectorizer

# 🔹 Load Model & Vectorizer
clf, vectorizer_tfidf = load_model()


# 🔹 Hugging Face API for Treatment Recommendations
HF_API_KEY = "KEY" # Replace with your actual API key

def get_treatment_recommendation(disease):
    """Query Hugging Face API for AI-powered treatment recommendations."""
    API_URL = "https://api-inference.huggingface.co/models/google/flan-t5-large"
    headers = {"Authorization": f"Bearer {HF_API_KEY}"}
    payload = {"inputs": f"What is the recommended treatment for {disease}?"}

    response = requests.post(API_URL, headers=headers, json=payload)

    if response.status_code == 200:
        result = response.json()
        if result and isinstance(result, list) and "generated_text" in result[0]:
            return result[0]["generated_text"]
        else:
            return "⚠️ No generated text in response."
    else:
        return f"⚠️ Error: {response.status_code} - {response.text}"

# 🔹 Streamlit UI
st.title("🩺 Disease Prediction & Treatment Recommendations")
st.write("Select your symptoms below to get a predicted disease and treatment suggestion.")

# 🔹 Load Symptoms Dataset
@st.cache_data
def load_symptom_data():
    data_path = os.path.join(drive_path, "processed_diseases-priority.csv")  # Ensure correct file path
    df = pd.read_csv(data_path)
    return df

df = load_symptom_data()

# 🔹 Extract Unique Symptoms
all_symptoms = set()
for symptoms in df["Symptoms"].astype(str).fillna(""):
    all_symptoms.update(symptoms.split(", "))

# 🔹 User Input: Multi-select Symptoms
selected_symptoms = st.multiselect("🩺 Select Symptoms:", sorted(all_symptoms))

if st.button("🔍 Predict Disease & Get Treatment"):
    if selected_symptoms:
        # 🔹 Format symptoms into a single input string
        symptoms_input = ", ".join(selected_symptoms)


        # 🔹 Predict disease using TF-IDF + Random Forest
        input_vector = vectorizer_tfidf.transform([symptoms_input])
        predicted_disease = clf.predict(input_vector)[0]

        # 🔹 Get AI-generated treatment recommendation
        treatment_recommendation = get_treatment_recommendation(predicted_disease)

        # 🔹 Display results
        st.success(f"🩺 **Predicted Disease:** {predicted_disease}")
        st.info(f"💊 **AI-Recommended Treatment:** {treatment_recommendation}")
    else:
        st.warning("⚠️ Please select at least one symptom!")


Overwriting app.py


## Testing the app

In [60]:
!npm install localtunnel
!streamlit run app.py --server.address=localhost &>/content/logs.txt &
!npx localtunnel --port 8501 & curl https://loca.lt/mytunnelpassword

⠙⠹⠸⠼⠴⠦⠧
up to date, audited 23 packages in 1s
⠧
⠧3 packages are looking for funding
⠧  run `npm fund` for details
⠧
2 high severity vulnerabilities

To address all issues (including breaking changes), run:
  npm audit fix --force

Run `npm audit` for details.
⠧34.71.24.72⠙your url is: https://plenty-plants-dig.loca.lt


## Creating the .py for deloying the app on steamlit community cloud

In [61]:
import streamlit as st
import pandas as pd
import joblib
import os
import requests

# 🔹 Set Google Drive path (Modify if necessary)
#drive_path = "/content/"


# 🔹 Load the trained model and vectorizer
@st.cache_resource
def load_model():
    model_path = os.path.join("disease_model.pkl")
    vectorizer_path = os.path.join("tfidf_vectorizer.pkl")

    if not os.path.exists(model_path) or not os.path.exists(vectorizer_path):
        st.error("🚨 Model or vectorizer not found! Please train and save them first.")
        return None, None

    clf = joblib.load(model_path,mmap_mode="r")
    vectorizer = joblib.load(vectorizer_path)
    return clf, vectorizer

# 🔹 Load Model & Vectorizer
clf, vectorizer_tfidf = load_model()


# 🔹 Hugging Face API for Treatment Recommendations
HF_API_KEY = st.secrets["HF_API_KEY"] #set API key on Streamlit Cloud, go to your app's Settings → Secrets

def get_treatment_recommendation(disease):
    """Query Hugging Face API for AI-powered treatment recommendations."""
    API_URL = "https://api-inference.huggingface.co/models/google/flan-t5-large"
    headers = {"Authorization": f"Bearer {HF_API_KEY}"}
    payload = {"inputs": f"What is the recommended treatment for {disease}?"}

    response = requests.post(API_URL, headers=headers, json=payload)

    if response.status_code == 200:
        result = response.json()
        if result and isinstance(result, list) and "generated_text" in result[0]:
            return result[0]["generated_text"]
        else:
            return "⚠️ No generated text in response."
    else:
        return f"⚠️ Error: {response.status_code} - {response.text}"

# 🔹 Streamlit UI
st.title("🩺 Disease Prediction & Treatment Recommendations")
st.write("Select your symptoms below to get a predicted disease and treatment suggestion.")

# 🔹 Load Symptoms Dataset
@st.cache_data
def load_symptom_data():
    data_path = os.path.join("processed_diseases-priority.csv")  # Ensure correct file path
    df = pd.read_csv(data_path)
    return df

df = load_symptom_data()

# 🔹 Extract Unique Symptoms
all_symptoms = set()
for symptoms in df["Symptoms"].astype(str).fillna(""):
    all_symptoms.update(symptoms.split(", "))

# 🔹 User Input: Multi-select Symptoms
selected_symptoms = st.multiselect("🩺 Select Symptoms:", sorted(all_symptoms))

if st.button("🔍 Predict Disease & Get Treatment"):
    if selected_symptoms:
        # 🔹 Format symptoms into a single input string
        symptoms_input = ", ".join(selected_symptoms)


        # 🔹 Predict disease using TF-IDF + Random Forest
        input_vector = vectorizer_tfidf.transform([symptoms_input])
        predicted_disease = clf.predict(input_vector)[0]

        # 🔹 Get AI-generated treatment recommendation
        treatment_recommendation = get_treatment_recommendation(predicted_disease)

        # 🔹 Display results
        st.success(f"🩺 **Predicted Disease:** {predicted_disease}")
        st.info(f"💊 **AI-Recommended Treatment:** {treatment_recommendation}")
    else:
        st.warning("⚠️ Please select at least one symptom!")


StreamlitSecretNotFoundError: No secrets found. Valid paths for a secrets.toml file or secret directories are: /root/.streamlit/secrets.toml, /content/.streamlit/secrets.toml

In [56]:
!ls -lh

total 1.1G
-rw-r--r--  1 root root 3.1K Apr  8 17:32 app.py
-rw-r--r--  1 root root 1.1G Apr  8 17:40 disease_model.pkl
drwx------  7 root root 4.0K Apr  8 15:09 gdrive
-rw-r--r--  1 root root  186 Apr  8 17:36 logs.txt
drwxr-xr-x 25 root root 4.0K Apr  8 15:23 node_modules
-rw-r--r--  1 root root   56 Apr  8 15:23 package.json
-rw-r--r--  1 root root 8.9K Apr  8 17:35 package-lock.json
-rw-r--r--  1 root root 3.5M Apr  8 17:00 processed_diseases-priority.csv
drwxr-xr-x  1 root root 4.0K Apr  4 13:38 sample_data
-rw-r--r--  1 root root  71K Apr  8 17:40 tfidf_vectorizer.pkl
